In [ ]:
/**
 * ============================================================================
 * HANDLING HIGH WRITE TRAFFIC — EXPLAINED LIKE WE'RE TALKING
 * ============================================================================
 *
 * Most apps mostly READ data (people browsing, scrolling, searching). But
 * some systems get absolutely hammered with WRITES instead: GPS pings
 * from millions of drivers, clicks on every page, server logs, sensor
 * readings, stock trades, orders — a constant firehose of new data.
 *
 * Every single write forces you to answer some uncomfortable questions:
 *   - When is it actually SAFE to tell the client "yep, saved!"?
 *   - What ORDER do these writes need to happen in?
 *   - What happens if a server dies mid-write?
 *   - How do we stop ONE busy part of the system from becoming the
 *     bottleneck for EVERYTHING?
 *
 * Here's the key mental model: READS are flexible. You can serve them
 * from a cache, from a replica, slightly stale, whatever — lots of
 * escape hatches. WRITES are not as flexible. If your product tells a
 * user "your write succeeded," you either need a real durable place for
 * that data, or you need to have HONESTLY decided that losing some data
 * sometimes is okay. No pretending otherwise.
 *
 *
 * ============================================================================
 * 1. WHY WRITES ARE HARDER TO SCALE THAN READS
 * ============================================================================
 *
 * -----------------------------------------------------------------------
 * 1.1 THE FUNDAMENTAL DIFFERENCE
 * -----------------------------------------------------------------------
 *
 * READS ARE EASY TO SCALE (just add more copies):
 *
 *   [Read Request] -> [Cache] -> [Replica 1]
 *                             -> [Replica 2]
 *                             -> [Replica N]
 *
 *   Need more read capacity? Add another cache layer, spin up more
 *   read-replicas, or serve slightly-old data if freshness isn't
 *   critical. All of this scales "horizontally" pretty easily.
 *
 * WRITES ARE HARDER (someone has to coordinate them):
 *
 *   [Write Request] -> [Leader/Primary] -> [Durable Queue] -> [Sharded Store]
 *                            (where exactly do we "safely acknowledge" this?)
 *
 *   Once you accept a write, SOMETHING has to order it, check for
 *   duplicates, validate it, and make it durable enough to match
 *   whatever promise you made to the user. That "something" could be a
 *   primary database, a Kafka partition, a Cassandra replica set, or a
 *   whole consensus group — but it's ALWAYS a specific, narrow place.
 *
 * KEY INSIGHT: the bottleneck isn't always "the database." It's whatever
 * is the NARROWEST part of your write path — a single leader, a busy
 * partition, a slow queue, disk sync speed, waiting for enough replicas
 * to confirm, or one downstream consumer falling behind. Scaling writes
 * means finding THAT narrow point and either splitting it up or moving
 * it — while being honest about what consistency or data you might give up.
 *
 * -----------------------------------------------------------------------
 * 1.2 QUESTIONS TO ASK BEFORE PICKING ANY TECHNOLOGY
 * -----------------------------------------------------------------------
 *
 *   Question                              | Why it matters
 *   -----------------------------------------|--------------------------------------
 *   Can we lose acknowledged writes?         | Decides: memory buffer vs durable
 *                                            | queue vs synchronous replication
 *   Does ORDER matter?                       | Do you need per-key ordering, a
 *                                            | single leader, or a partitioned log?
 *   When must READS see the write?            | Strongly consistent vs eventually
 *                                            | consistent (analytics can lag a bit)
 *   What's the natural "partition key"?       | Drives sharding, hot-key risk, and
 *                                            | how painful cross-shard ops will be
 *   Is traffic steady or spiky?               | Determines buffering, backpressure,
 *                                            | and load-shedding needs
 *
 * -----------------------------------------------------------------------
 * 1.3 JUST HOW BIG CAN THIS PROBLEM GET? (a real example)
 * -----------------------------------------------------------------------
 *
 * Say you're running an IoT platform with 10 MILLION devices, each
 * sending a reading every 5 seconds, and each message is 500 bytes.
 *
 *   10,000,000 devices / 5 seconds  = 2,000,000 writes PER SECOND
 *   2,000,000 writes * 500 bytes    = 1 GB of data ingested PER SECOND
 *   over a full day                 = 86 TB of data PER DAY
 *
 * A well-tuned regular relational database might handle maybe TENS OF
 * THOUSANDS of simple inserts per second — nowhere close to 2 MILLION
 * per second as one continuous stream of single inserts. That massive
 * GAP between "what one database can do" and "what you actually need"
 * is exactly why all the techniques in this file exist.
 *
 * -----------------------------------------------------------------------
 * 1.4 DIFFERENT SYSTEMS, DIFFERENT REQUIREMENTS
 * -----------------------------------------------------------------------
 *
 *   System                | Writes/sec | What matters most
 *   ------------------------|--------------|---------------------------------------
 *   Logging/Monitoring       | 1M+          | Append-only, can tolerate SOME loss
 *   Analytics/Clickstream    | 100K+        | Batch-friendly, eventual consistency OK
 *   IoT Platforms             | 1M+          | High volume, tiny payloads, spread out
 *   Financial Trading         | 100K+        | Low latency, ordering matters, ZERO loss
 *   Social Media Posts        | 10K+         | Mixed with heavy reads, needs consistency
 *   Ad Tech Bidding            | 1M+          | Real-time, sub-100ms decisions
 *
 *   Big lesson: a LOGGING system can shrug off occasionally losing a
 *   record. A TRADING system absolutely cannot lose even one acknowledged
 *   transaction. These different tolerances lead to completely different
 *   architectures — there's no one-size-fits-all answer.
 *
 * -----------------------------------------------------------------------
 * 1.5 THE DURABILITY VS. SPEED TRADE-OFF
 * -----------------------------------------------------------------------
 * How FAST you can say "write succeeded!" depends entirely on how much
 * safety you're willing to promise. Think of it as a sliding scale:
 *
 *   Level | Method                       | Speed      | Survives...
 *   -------|--------------------------------|--------------|------------------------------
 *   1      | Memory only                     | ~1 microsecond | NOTHING (crash = data gone)
 *   2      | Async disk write                 | ~100 microsecond | The PROCESS crashing
 *                                              |            | (not the whole machine)
 *   3      | Sync disk write (fsync)           | ~1-10 ms    | The MACHINE crashing
 *   4      | Sync write + replica confirms      | ~10-50 ms   | Losing that WHOLE machine
 *   5      | Multi-region sync                  | ~50-200 ms  | Losing an ENTIRE datacenter
 *
 *   PICTURE (the spectrum):
 *
 *     [Memory Only] -> [Async Disk] -> [Sync Disk] -> [Sync+Replica] -> [Multi-Region]
 *       fastest, riskiest                                          slowest, safest
 *
 *   A metrics system might happily accept Level 2 — losing a few
 *   seconds of metrics during a crash is a shrug. A banking system needs
 *   Level 4 or 5 — losing even ONE transaction is unacceptable. Knowing
 *   where YOUR system belongs on this scale is step one of the whole design.
 *
 *
 * ============================================================================
 * 2. THE WRITE-SCALING TOOLKIT (the big picture)
 * ============================================================================
 *
 * There are 6 main tools. In practice you'll combine several of them
 * together, not pick just one.
 *
 *   Technique             | What it does                              | When to reach for it
 *   ------------------------|-----------------------------------------------|--------------------------------------
 *   Batching                 | Combines many small writes into one bigger one | Almost always your FIRST move
 *   Async Processing          | Separates "accept the write" from "actually  | When instant confirmation
 *                            | do the write"                                 | isn't required
 *   Sharding                  | Spreads writes across multiple databases     | When ONE database maxes out
 *   Write-Optimized Storage    | Uses storage engines built for writing fast  | When write speed is THE bottleneck
 *   Event Sourcing / CQRS      | Fully separates the "write model" from the   | Complex domains, very different
 *                              | "read model"                                 | read vs write patterns
 *   Backpressure                | Protects the system when it's overloaded    | ALWAYS (this is your safety net)
 *
 *   Good default ORDER to think in: start with Batching (cheap, simple),
 *   then add Async Processing if a delay is acceptable, then Sharding
 *   once one node truly can't keep up anymore.
 *
 *
 * ============================================================================
 * 3. BATCHING AND BUFFERING — "stop writing one at a time"
 * ============================================================================
 *
 * -----------------------------------------------------------------------
 * 3.1 WHY BATCHING WORKS (the overhead problem)
 * -----------------------------------------------------------------------
 * Every single database write has a bunch of FIXED overhead attached to
 * it, no matter how small the actual data is: network travel time,
 * parsing the query, starting a transaction, and syncing to disk.
 *
 *   ONE INDIVIDUAL WRITE (example):
 *     Network round-trip:     2ms
 *     Parse the query:        0.1ms
 *     Begin transaction:      0.1ms
 *     Actually write data:    0.5ms   <- this is the ONLY "real" work
 *     Commit + fsync:         2ms
 *     TOTAL:                  ~5ms
 *
 *   Notice: the ACTUAL data write is just 0.5ms — the other 4.5ms is
 *   pure overhead! Do this 1,000 times = 5 FULL SECONDS just for
 *   overhead, to insert 1,000 tiny rows.
 *
 *   NOW BATCH IT (1,000 rows in ONE go):
 *     Network round-trip:     2ms   (paid ONCE)
 *     Parse the query:        0.2ms (paid ONCE)
 *     Begin transaction:      0.1ms (paid ONCE)
 *     Write all 1000 rows:    50ms
 *     Commit + fsync:         2ms   (paid ONCE)
 *     TOTAL:                  ~55ms
 *
 *   1,000 rows in 55ms instead of 5,000ms — nearly 100x faster! The
 *   exact numbers will vary by system, but the REASON is always the
 *   same: pay the fixed overhead ONCE for many rows, instead of once
 *   PER row.
 *
 * -----------------------------------------------------------------------
 * 3.2 CLIENT-SIDE BATCHING (buffer in your own app first)
 * -----------------------------------------------------------------------
 * Instead of writing to the database immediately, hold new writes in
 * memory temporarily, then flush them all together.
 *
 * PICTURE:
 *
 *   [App writes 1, 2, 3, ...N] -> [In-Memory Buffer]
 *                                       |
 *                          flush when EITHER of these happens:
 *                            - buffer hits a SIZE limit (e.g. 1000 items)
 *                            - buffer hits a TIME limit (e.g. 100ms)
 *                                       |
 *                                       v
 *                                 [Database]
 *
 *   Having BOTH a size limit and a time limit matters: the size limit
 *   keeps big bursts efficient, and the time limit makes sure you don't
 *   wait forever for a full batch during quiet periods.
 *
 *   TRADE-OFF: latency for throughput. A single write finishes
 *   instantly. A batched write might sit in the buffer for up to your
 *   time limit before actually being saved. Totally fine for analytics
 *   or logging — probably NOT fine for processing a payment.
 *
 * -----------------------------------------------------------------------
 * 3.3 DATABASE-SIDE BATCHING (let the DB help you)
 * -----------------------------------------------------------------------
 * Different databases give you different "bulk write" tools:
 *
 *   Method              | What it looks like            | Improvement
 *   ----------------------|----------------------------------|----------------------------------
 *   Individual INSERTs      | One row per statement            | Simple, but tons of overhead
 *   Multi-value INSERT       | Many rows in ONE statement        | Fewer round trips, less overhead
 *   COPY command (Postgres)   | A dedicated bulk-load path        | Skips per-row INSERT overhead
 *                            |                                    | entirely — very efficient
 *
 *   The exact speedup varies, but the pattern is always: fewer round
 *   trips + a dedicated bulk path = better throughput.
 *
 * -----------------------------------------------------------------------
 * 3.4 THE "WRITE-BEHIND" PATTERN (fast now, save for real later)
 * -----------------------------------------------------------------------
 * For MAXIMUM speed, write to something super fast first, and let a
 * background process handle the "real," slower persistence later.
 *
 * PICTURE:
 *
 *   FAST PATH (synchronous):
 *     [Application] --write (super fast)--> [Redis, in-memory]
 *                   <---return immediately----
 *
 *   SLOW PATH (asynchronous, happens in background):
 *     [Background Worker] --reads batches from--> [Redis]
 *                          --bulk writes to------> [Database]
 *
 *   The app writes to Redis (very fast) and immediately tells the user
 *   "done!" A worker quietly reads from Redis, batches things up, and
 *   writes them to the real database at its own pace. Redis absorbs
 *   traffic spikes because it's much faster than most databases.
 *
 *   THE RISK: if Redis crashes BEFORE the worker gets a chance to
 *   persist the data, you've just lost writes you already told the user
 *   were "done." Redis's own persistence/replication features reduce
 *   this risk, but they DON'T make this pattern as safe as a real
 *   durable queue or a database commit. Only use write-behind if your
 *   product can tolerate that risk — or put a proper durable log/queue
 *   in front of the acknowledgment instead.
 *
 * -----------------------------------------------------------------------
 * 3.5 SENDING FEWER WRITES IN THE FIRST PLACE
 * -----------------------------------------------------------------------
 * Batching makes each write cheaper. But you can ALSO just... generate
 * fewer writes to begin with. For metrics, clickstream, and IoT data, a
 * huge chunk of the raw data doesn't carry much INDIVIDUAL value — you
 * can shrink it before it even reaches storage.
 *
 * PICTURE:
 *
 *   [1000 raw sensor readings/min] --Edge Aggregation--> [1 rollup row/min: min/max/avg/count]
 *
 *   THREE TECHNIQUES:
 *
 *     Technique         | What it does                                  | Cost
 *     --------------------|--------------------------------------------------|----------------------------------
 *     Pre-aggregation       | Combine many raw events into ONE summary          | You lose per-event detail
 *                          | (e.g. per-minute count/sum/min/max) before saving |
 *     Sampling               | Only keep a slice of events (e.g. 1 in 100),      | Some statistical error, might
 *                          | then scale the numbers up later                    | miss rare events entirely
 *     Rollups/Downsampling    | Keep RECENT data at full detail, compact OLDER   | Old data loses precision
 *                          | data into coarser buckets over time               | over time
 *
 *   Example: a metrics tool like StatsD has clients constantly emitting
 *   counters — but the agent aggregates them in memory and only flushes
 *   ONE summary per time interval, so storage sees orders of magnitude
 *   fewer writes than the app actually generated.
 *
 *   THE JUDGMENT CALL: what are you ALLOWED to lose? Aggregation and
 *   sampling are great for dashboards and trend graphs — but terrible
 *   for an audit log or a billing meter, where EVERY single event has to
 *   survive intact.
 *
 *
 * ============================================================================
 * 4. ASYNCHRONOUS PROCESSING — "accept now, actually save later"
 * ============================================================================
 *
 * Batching changes HOW writes happen. Async processing changes WHEN
 * they happen. Instead of writing straight to the database, you write
 * to a message queue instead, and a SEPARATE worker actually does the
 * database write.
 *
 * -----------------------------------------------------------------------
 * 4.1 THE BASIC PATTERN
 * -----------------------------------------------------------------------
 *
 * PICTURE:
 *
 *   [API Server 1] --publish--> [Message Queue: Kafka/SQS/RabbitMQ] --consume--> [Worker 1] --write--> [Database]
 *   [API Server 2] --publish-->        (durable buffer)             --consume--> [Worker 2] --write-->
 *   [API Server N] --publish-->                                     --consume--> [Worker N] --write-->
 *
 * WHY THIS HELPS:
 *   - Decoupling: the API server doesn't wait for the database — it
 *     pushes to the queue and moves on immediately. If the database is
 *     briefly slow, the API server doesn't even notice.
 *   - Buffering: the queue soaks up sudden traffic spikes. Workers drain
 *     it steadily; the database never sees the raw spike directly.
 *   - Scalability: need to catch up? Just add more workers. The queue
 *     handles the coordination.
 *   - Resilience: if the database goes down entirely AFTER something
 *     was safely published to the durable queue, that write just waits
 *     patiently until the database is back — instead of just failing.
 *
 * -----------------------------------------------------------------------
 * 4.2 CHOOSING THE RIGHT QUEUE
 * -----------------------------------------------------------------------
 *
 *   Queue           | Strength                       | Ordering       | Best for
 *   -------------------|-----------------------------------|-------------------|----------------------------------
 *   Kafka                | High-throughput partitioned log    | Per partition     | High-throughput streaming, logs
 *   RabbitMQ              | Flexible, routing-heavy messaging  | Per queue          | Work queues, complex routing
 *   Amazon SQS             | Managed, elastic, low-maintenance   | FIFO optional      | Serverless, managed setups
 *   Redis Streams          | Very low latency (in-memory)         | Per stream          | Lightweight pipelines already
 *                        |                                     |                    | using Redis
 *
 *   Kafka is popular for high throughput because each "partition" is an
 *   INDEPENDENT ordered log — with enough partitions and consumers, you
 *   can process a lot of messages in parallel while still keeping order
 *   WITHIN each partition.
 *
 * -----------------------------------------------------------------------
 * 4.3 HOW YOU CONSUME MATTERS TOO
 * -----------------------------------------------------------------------
 *
 *   A) COMPETING CONSUMERS (just scale out, order not guaranteed)
 *
 *      PICTURE:
 *        [Queue] -> [Consumer 1]
 *               \-> [Consumer 2]
 *               \-> [Consumer N]
 *        (each message goes to ONE consumer, but no ordering guarantee
 *         across different messages — and duplicates are possible on
 *         failure/retry)
 *
 *   B) PARTITIONED CONSUMERS (ordered PER KEY, and still scalable)
 *
 *      PICTURE:
 *        [Kafka Topic]
 *          Partition 0 (user_id % 3 == 0) -> [Consumer 0]
 *          Partition 1 (user_id % 3 == 1) -> [Consumer 1]
 *          Partition 2 (user_id % 3 == 2) -> [Consumer 2]
 *
 *      All messages for the SAME key (e.g. same user_id) go to the SAME
 *      partition, always processed by ONE consumer at a time — giving
 *      you ordering WITHIN a partition, while still scaling ACROSS
 *      partitions.
 *
 * -----------------------------------------------------------------------
 * 4.4 HANDLING FAILURES: MAKE PROCESSING IDEMPOTENT
 * -----------------------------------------------------------------------
 * A worker might crash AFTER processing a message but BEFORE
 * acknowledging it. The queue, not knowing better, will redeliver that
 * message to someone else — meaning it gets processed TWICE.
 *
 * FIX: make processing "idempotent" — handling the same message twice
 * should produce the SAME result as handling it once.
 *
 * PICTURE (the check-before-processing flow):
 *
 *   [Message arrives] -> Already processed (by ID)?
 *        YES -> skip, just return success
 *        NO  -> process it -> store the result + mark this message ID
 *               as done -> acknowledge
 *
 *   Keep a small table (or a Redis set) of "processed_events" — before
 *   doing any work, check if you've already seen this message's ID.
 *   Simple pattern, makes your whole pipeline resilient to duplicates.
 *
 * -----------------------------------------------------------------------
 * 4.5 WATCH "CONSUMER LAG" — the hidden failure mode
 * -----------------------------------------------------------------------
 * A durable queue absorbs spikes nicely — but it can HIDE a growing
 * problem: if workers are consuming SLOWER than producers are writing,
 * the backlog just keeps growing, and your stored data falls further
 * and further behind reality.
 *
 * "Consumer lag" = the gap between the newest message written, and the
 * newest message a consumer group has actually finished processing.
 *
 *   PICTURE:
 *
 *     latest offset written:     1,000,000
 *     offset consumer has done:    940,000
 *     ------------------------------------
 *     LAG = 60,000 messages behind
 *
 *   Rising lag = consumers can't keep up. Fixes: add more consumers (up
 *   to the number of partitions — one partition can only be handled by
 *   one consumer at a time in a group), increase batch size on writes,
 *   or apply backpressure further upstream. Flat, low lag = healthy pipeline.
 *
 *   Interview tip: mentioning "I'd alert on consumer lag and partition
 *   count" shows you think of this pipeline as something you actively
 *   OPERATE, not just something you draw once and forget.
 *
 *
 * ============================================================================
 * 5. WRITE-OPTIMIZED STORAGE — using databases built for writing fast
 * ============================================================================
 *
 * General databases like PostgreSQL and MySQL try to be decent at BOTH
 * reads and writes. But if writes are your bottleneck, it helps to
 * understand storage engines built SPECIFICALLY to write fast.
 *
 * -----------------------------------------------------------------------
 * 5.1 B-TREES vs LSM TREES
 * -----------------------------------------------------------------------
 *
 * B-TREES (used by PostgreSQL, MySQL) — optimized for READS:
 *
 *   PICTURE (a single write, step by step):
 *
 *     [Write Request] -> find the right page in the tree
 *                      -> read that page from disk
 *                      -> modify the page
 *                      -> write the page back to disk
 *                      -> update any INDEX pages too
 *
 *   This means a single logical write might touch multiple data pages
 *   AND index pages AND the write-ahead log — lots of scattered, random
 *   reads/writes. This is called "write amplification," and it gets
 *   worse with more indexes.
 *
 * LSM TREES (Log-Structured Merge Trees, used by Cassandra, RocksDB) —
 * optimized for WRITES:
 *
 *   PICTURE:
 *
 *     [Write] -> [Memtable: in-memory buffer]
 *                       | (when full, flush to disk)
 *                       v
 *                [Level 0 SSTable: on disk, sorted]
 *                       | (background compaction merges these over time)
 *                       v
 *                [Level 1 SSTable] -> [Level 2 SSTable: largest level]
 *
 *   New writes just go into memory first (fast!), then get flushed to
 *   disk as a sorted file. Background processes quietly merge and clean
 *   up these files over time ("compaction"). The write path itself is
 *   mostly just APPENDING — which is exactly why LSM systems are so
 *   good at high-volume ingestion.
 *
 * -----------------------------------------------------------------------
 * 5.2 THE TRADE-OFF
 * -----------------------------------------------------------------------
 *
 *   Aspect              | B-Tree (PostgreSQL)          | LSM Tree (Cassandra)
 *   ----------------------|----------------------------------|---------------------------------------
 *   Write path              | In-place updates, page splits      | Append-oriented (memtables -> SSTables)
 *   Read path                | Strong, indexed point/range reads   | May need to check MULTIPLE files/levels
 *   Write amplification       | Can be high with many indexes       | Can be significant during compaction
 *   Space usage                | Usually lower                       | Higher, especially during compaction
 *   Best for                    | Mixed read/write, transactions       | Write-heavy, mostly-append workloads
 *
 *   LSM reads may need to check several files across several "levels"
 *   before finding your data. Bloom filters, caching, and smart
 *   compaction strategies help — but the fundamental trade-off is:
 *   MORE read work and compaction overhead, in exchange for a MUCH
 *   friendlier write path.
 *
 * -----------------------------------------------------------------------
 * 5.3 REAL LSM-BASED DATABASES
 * -----------------------------------------------------------------------
 *
 *   Database    | Type              | Best used for
 *   --------------|----------------------|--------------------------------------
 *   Cassandra      | Wide-column          | Time-series, IoT, very high write volume
 *   ScyllaDB        | Wide-column          | Cassandra-compatible, higher performance
 *   RocksDB          | Embedded key-value    | A storage engine other systems build on
 *   ClickHouse        | Columnar              | Analytics, log aggregation
 *   InfluxDB           | Time-series            | Metrics, monitoring, IoT
 *
 *   When your write requirements outgrow PostgreSQL/MySQL, these belong
 *   on your shortlist.
 *
 * -----------------------------------------------------------------------
 * 5.4 APPEND-ONLY LOGS — the simplest write-friendly shape of all
 * -----------------------------------------------------------------------
 *
 * PICTURE:
 *
 *   [Event 1][Event 2][Event 3][Event 4]...  <- write pointer moves forward,
 *                                                new writes just get ADDED to the end
 *
 * An append-only log just writes records one after another, sequentially,
 * to the end of a file. No modifying old data. Any indexing or query
 * optimization happens SEPARATELY from the writing itself.
 *
 * This is literally how Kafka topics work internally. It's how database
 * write-ahead logs (WALs) work. It's the foundation of event sourcing
 * (covered below). The downside: reading requires either scanning the
 * whole log or building an external index. Great fit for logging, event
 * capture, and replication.
 *
 *
 * ============================================================================
 * 6. SHARDING FOR WRITES — splitting the load across many databases
 * ============================================================================
 *
 * Once you've batched, gone async, and picked a good storage engine, a
 * SINGLE database still has a ceiling. To go beyond that, you spread
 * writes across MULTIPLE databases — this is "sharding."
 *
 * -----------------------------------------------------------------------
 * 6.1 THE BASIC ARCHITECTURE
 * -----------------------------------------------------------------------
 *
 * PICTURE (4-shard setup, keyed on user_id):
 *
 *   [Application] -> [Shard Router] -> user_id % 4 == 0 -> [Shard 0]
 *                                    -> user_id % 4 == 1 -> [Shard 1]
 *                                    -> user_id % 4 == 2 -> [Shard 2]
 *                                    -> user_id % 4 == 3 -> [Shard 3]
 *
 *   With 4 well-balanced shards, each database only handles ~25% of the
 *   writes — total capacity can approach 4x a single database (assuming
 *   no shared bottlenecks and even distribution). That's TRUE horizontal
 *   scaling for writes.
 *
 * -----------------------------------------------------------------------
 * 6.2 PICKING A GOOD SHARD KEY (this decision matters a LOT)
 * -----------------------------------------------------------------------
 *
 *   BAD EXAMPLE — sharding by TIMESTAMP:
 *
 *     [ALL current writes] -> all land on the "today" shard
 *     [Yesterday's shard]  -> sits mostly idle now
 *
 *     Every NEW write piles onto whichever shard owns "right now" — a
 *     classic "hot shard" problem.
 *
 *   GOOD EXAMPLE — sharding by USER ID:
 *
 *     User A -> Shard 1
 *     User B -> Shard 2
 *     User C -> Shard 3
 *
 *     (Much more evenly spread out, since users are naturally different
 *     from each other)
 *
 *   COMPARISON TABLE:
 *
 *     Shard key       | Distribution              | Pros                    | Cons
 *     -------------------|------------------------------|----------------------------|--------------------------------------
 *     user_id             | Usually even                | Keeps a user's data together | A "viral" user can create a hot shard
 *     timestamp             | Uneven (all "now" = 1 shard)| Easy time-range queries      | The newest shard gets ALL the writes
 *     random/UUID             | Usually even                | Very even load               | Related data ends up scattered
 *     Composite key             | Controllable                | Balances multiple concerns    | More complex routing logic
 *
 *   For time-series data, a COMPOSITE key like (device_id, time_bucket)
 *   works great: device_id picks the SHARD (spreading writes evenly),
 *   while time_bucket groups a single device's data together WITHIN its
 *   shard (great for efficient time-range queries).
 *
 * -----------------------------------------------------------------------
 * 6.3 CONSISTENT HASHING — adding shards without moving EVERYTHING
 * -----------------------------------------------------------------------
 *
 * With naive "key % number_of_shards" hashing, adding just ONE new shard
 * shuffles almost EVERYTHING around:
 *
 *   PICTURE (naive modulo):
 *
 *     3 shards -> add 1 more -> 4 shards
 *     result: ~75% of ALL keys have to MOVE to a different shard!
 *
 *   With CONSISTENT HASHING, adding one shard only moves a SMALL slice:
 *
 *     3 shards -> add 1 more -> 4 shards
 *     result: only ~25% of keys move (roughly 1/4 moves to the new shard,
 *     everything else stays put)
 *
 *   HOW IT WORKS: imagine all your shards placed around a circular
 *   "ring." Each key hashes to a point on that ring and belongs to the
 *   NEXT shard clockwise from it. Add a new shard, and it just grabs a
 *   slice of the ring from its neighbor — only keys in THAT slice move.
 *
 *   ONE WRINKLE: if each shard only occupies ONE point on the ring, the
 *   new shard steals its ENTIRE slice from just ONE unlucky neighbor,
 *   making load uneven. Production systems fix this with "virtual
 *   nodes" — giving each REAL shard MANY points scattered around the
 *   ring, so a new shard takes small slices from MANY neighbors at
 *   once, spreading the moved keys out evenly.
 *
 * -----------------------------------------------------------------------
 * 6.4 THE CROSS-SHARD PROBLEM
 * -----------------------------------------------------------------------
 * Sharding creates a brand new headache: what if ONE operation needs to
 * touch data on TWO DIFFERENT shards?
 *
 * PICTURE (transferring $100 between users on different shards):
 *
 *   [Transfer $100] -> Debit User A (Shard 1) + Credit User B (Shard 3)
 *   THIS MUST BE ATOMIC — either BOTH happen, or NEITHER does!
 *   (if the debit succeeds but the credit fails, money just vanished)
 *
 *   APPROACHES:
 *
 *     Approach                  | Consistency | Performance              | Complexity
 *     -----------------------------|----------------|-----------------------------|--------------
 *     Two-Phase Commit (2PC)         | Strong          | Slow (locks span shards)     | High
 *     Saga Pattern                    | Eventual        | Good                          | Medium
 *     Avoid cross-shard writes           | N/A             | Best (by far)                  | Needs good
 *                                    |                |                              | data modeling
 *
 *   BEST SOLUTION: design your data so the MOST COMMON write stays
 *   inside a SINGLE shard (e.g. an account and its own ledger entries
 *   share the same shard key). Transfers between UNRELATED users
 *   naturally cross shard boundaries — that's when you actually reach
 *   for sagas, 2PC, or a dedicated ledger design.
 *
 * -----------------------------------------------------------------------
 * 6.5 LEADER-BASED vs LEADERLESS WRITES (within one shard's group)
 * -----------------------------------------------------------------------
 * Sharding decides WHICH node owns a key. A SEPARATE question is how
 * writes get accepted and copied WITHIN that group of nodes.
 *
 *   LEADER-BASED (PostgreSQL primary/replica, MySQL, Kafka partitions):
 *
 *     [Write] -> [Leader] -> copies to -> [Replica] [Replica]
 *
 *     Every write for a shard goes through ONE leader. Clean, simple,
 *     easy to reason about ordering — but that leader is ALSO your
 *     write ceiling, and if it dies, writes PAUSE until a new leader
 *     gets elected.
 *
 *   LEADERLESS / QUORUM-BASED (Cassandra, DynamoDB, Riak):
 *
 *     [Write] -> sent to -> [Node A] [Node B] [Node C]
 *              -> considered successful once "W" of them acknowledge
 *
 *     No single node to overload or fail over — write AVAILABILITY
 *     stays high even if one node is having a bad day.
 *
 *   COMPARISON:
 *
 *     Aspect          | Leader-Based                          | Leaderless (Quorum)
 *     -------------------|------------------------------------------|--------------------------------------------
 *     Write path            | ALL writes go through one leader per shard | Sent to N replicas, wait for W acks
 *     Ordering                | Clean, single natural order                | No global order — conflicts resolved LATER
 *     Failure impact            | Writes PAUSE during leader election          | Writes keep flowing while quorum reachable
 *     Consistency                | Strong, per shard                             | Tunable via R and W settings
 *
 *   TUNING TIP: with quorum-based systems, R (reads needed) + W (writes
 *   needed) > N (total replicas) guarantees a read will always overlap
 *   with the most recent write. Smaller R/W values trade freshness for
 *   speed and availability. Conflicting concurrent writes still need
 *   resolving — usually "last write wins" by timestamp, or version vectors.
 *
 *
 * ============================================================================
 * 7. EVENT SOURCING AND CQRS — storing the "story," not just the ending
 * ============================================================================
 *
 * -----------------------------------------------------------------------
 * 7.1 EVENT SOURCING — store what HAPPENED, not just where things ended up
 * -----------------------------------------------------------------------
 *
 * TRADITIONAL WAY (mutable state — update the record in place):
 *
 *   [Update 1: balance=100] -> [Update 2] -> [Update 3: balance=150]
 *   OLD HISTORY IS LOST — you only ever see the FINAL number, 150.
 *
 * EVENT SOURCING (store the events themselves, forever):
 *
 *   [UserCreated: balance=0] -> [DepositMade: +100] -> [WithdrawalMade: -20] -> [DepositMade: +70]
 *   "Replay" all these events -> Balance = 0 + 100 - 20 + 70 = 150
 *   COMPLETE HISTORY IS PRESERVED — you get a built-in audit trail for FREE.
 *
 *   Sounds slow if you replayed events on EVERY read — and it would be!
 *   In practice you keep separate, read-optimized "projections" that get
 *   updated as new events roll in. But the raw EVENTS remain the one
 *   true source of truth.
 *
 * WHY THIS HELPS WRITE PERFORMANCE:
 *
 *   Aspect          | Traditional (mutate in place)   | Event Sourcing (append-only)
 *   -------------------|-------------------------------------|-----------------------------------
 *   Write operation      | Update in place (random I/O)          | Append to a log (sequential I/O)
 *   Concurrency            | Conflicts on the SAME row directly     | Conflicts caught during validation
 *   Write speed              | Moderate                                | Very fast
 *   History                   | Lost after each update                  | Complete audit trail, forever
 *   Debugging                   | "Wait, what actually happened here?"     | Just replay events to any past point!
 *
 *   Important: event sourcing doesn't ELIMINATE conflicts — it just
 *   MOVES them, to "command validation" time. Example rule: "only append
 *   this new event if the account is still at version 42" (optimistic
 *   concurrency check).
 *
 * -----------------------------------------------------------------------
 * 7.2 CQRS (COMMAND QUERY RESPONSIBILITY SEGREGATION)
 * -----------------------------------------------------------------------
 * Event sourcing pairs naturally with CQRS, which means: totally
 * separate the WRITE side of your system from the READ side.
 *
 * PICTURE:
 *
 *   WRITE SIDE:
 *     [Write Request] -> [Command Service: validates] -> [Event Store: append-only log]
 *
 *   [Event Processor] subscribes to new events, and updates:
 *     -> [Read Model 1: User Profiles, in PostgreSQL]
 *     -> [Read Model 2: Analytics, in ClickHouse]
 *     -> [Read Model 3: Search Index, in Elasticsearch]
 *
 *   READ SIDE:
 *     [Read Request] -> [Query Service] -> whichever read model fits best
 *
 *   The big win: each MODEL is optimized independently. Writes stay
 *   fast (just appending to the event log). Reads get their OWN
 *   tailored storage for their OWN query pattern. You scale each side
 *   separately — the cost is "projection lag" (read models are slightly
 *   behind the write log) and extra operational complexity.
 *
 * -----------------------------------------------------------------------
 * 7.3 WHEN TO USE EVENT SOURCING / CQRS
 * -----------------------------------------------------------------------
 *
 *   GOOD FIT:
 *     - audit requirements (finance, healthcare, compliance)
 *     - complex domain logic with lots of possible state transitions
 *     - you genuinely need SEVERAL differently-shaped read models
 *     - high write throughput, and eventual consistency is acceptable
 *     - event-driven architectures with multiple downstream consumers
 *
 *   POOR FIT:
 *     - simple CRUD apps
 *     - every read needs STRONG, immediate consistency
 *     - small scale where the extra complexity just isn't worth it
 *     - your team isn't familiar with event-driven patterns yet
 *
 *
 * ============================================================================
 * 8. BACKPRESSURE AND RATE LIMITING — your safety net under overload
 * ============================================================================
 *
 * Every system has a limit. When traffic goes past it, you have exactly
 * two options: crash badly, or degrade GRACEFULLY. This section is
 * about making sure you pick "gracefully."
 *
 * -----------------------------------------------------------------------
 * 8.1 THE OVERLOAD PROBLEM (what happens with NO flow control)
 * -----------------------------------------------------------------------
 *
 * PICTURE (a cascade failure):
 *
 *   [Traffic spike: 10,000 writes/sec] -> [Server capacity: only 5,000/sec]
 *          -> internal queue keeps growing and growing
 *          -> memory gets exhausted
 *          -> CRASH
 *          -> every in-flight request is lost, AND the traffic that
 *             caused this now slams into your OTHER servers too
 *
 * -----------------------------------------------------------------------
 * 8.2 RATE LIMITING — capping how much ONE caller can send
 * -----------------------------------------------------------------------
 * Rate limiting is mainly about FAIRNESS and abuse-prevention: "this
 * specific caller is over their quota." It's NOT the same thing as load
 * shedding ("the whole SYSTEM is over capacity") — though it helps
 * during overload too.
 *
 * PICTURE:
 *
 *   [Requests] -> [Rate Limiter] -> under limit -> Accept
 *                                -> over limit  -> 429 Too Many Requests
 *
 *   COMMON ALGORITHMS:
 *
 *     Algorithm         | Behavior                       | Use case
 *     ---------------------|-------------------------------------|---------------------------
 *     Token Bucket           | Allows short bursts up to a limit     | API rate limiting (most common!)
 *     Leaky Bucket             | Smooths output, no bursts allowed      | Steady, even processing
 *     Fixed Window               | Just counts per time window             | Simple quota tracking
 *     Sliding Window                | Rolling count, smoother behavior         | More accurate limiting
 *
 *   TOKEN BUCKET example: tokens fill up steadily (say, 100/sec) up to
 *   a max (say, 500). Every request uses ONE token. This lets a short
 *   burst use up to 500 requests all at once, while still enforcing an
 *   AVERAGE rate of 100/sec over time.
 *
 * -----------------------------------------------------------------------
 * 8.3 BACKPRESSURE — telling COOPERATIVE producers to slow down
 * -----------------------------------------------------------------------
 * Instead of just accepting unlimited work, the system tells whoever is
 * SENDING data: "slow down," "retry later," "send smaller batches," or
 * "just pause for now."
 *
 * PICTURE:
 *
 *   [Producer] -> [Queue] -> [Consumer] -> [Database]
 *                     ^
 *        when queue fills up, send a backpressure signal
 *              back to the producer to slow down
 *
 *   MECHANISMS:
 *
 *     Mechanism         | How it works
 *     ---------------------|----------------------------------------------------
 *     Blocking                | Producer literally BLOCKS/pauses when queue is full
 *     Credits                  | Consumer hands out "credits"; producer pauses
 *                          | once it runs out of credits
 *     HTTP 503                  | Server explicitly says "Service Unavailable,
 *                          | try again in X seconds" (Retry-After header)
 *     TCP flow control            | The receiver advertises how much more it can
 *                          | accept right now (a "window size")
 *
 *   THE KEY RULE: accepting work you CANNOT finish is worse than slowing
 *   down or REJECTING it early. A bounded queue that pushes back is
 *   safer than an unbounded queue that just crashes later.
 *
 * -----------------------------------------------------------------------
 * 8.4 LOAD SHEDDING — deciding WHAT to drop when you have to drop something
 * -----------------------------------------------------------------------
 *
 * PICTURE:
 *
 *   [Request] -> is server overloaded?
 *       NO  -> process it normally
 *       YES -> check priority ->
 *                HIGH priority -> process it anyway
 *                LOW priority  -> shed/drop it, return 503
 *
 *   STRATEGIES:
 *
 *     Strategy               | How it works                         | Trade-off
 *     --------------------------|-------------------------------------------|-----------------------------------
 *     Random                     | Drop random requests                        | Fair, but wasteful
 *     Age/deadline-based           | Drop requests that are already too old       | They may have timed out anyway
 *     Priority                      | Drop LOW-priority requests first              | Protects your important traffic
 *     Client-based                    | Drop requests from clients over quota           | Fair to clients behaving well
 *
 *   The right strategy depends on the product: a payment system might
 *   prioritize finishing IN-PROGRESS transactions over accepting new
 *   ones; an analytics pipeline might drop events from its BIGGEST
 *   clients first, just to keep coverage fair across everyone else.
 *
 * -----------------------------------------------------------------------
 * 8.5 GRACEFUL DEGRADATION — do LESS, instead of doing NOTHING
 * -----------------------------------------------------------------------
 *
 * PICTURE:
 *
 *   NORMAL MODE:   Write to DB + Update Cache + Send to Analytics + Trigger Notifications
 *   DEGRADED MODE: Write to DB + Update Cache + (skip Analytics) + (skip Notifications)
 *
 *   Under heavy load, drop the LESS essential steps (analytics,
 *   notifications) and keep only the CORE operation running. Users still
 *   get the important part, even if some "nice to have" side effects get
 *   delayed or skipped.
 *
 *
 * ============================================================================
 * 9. PUTTING IT ALL TOGETHER — one full architecture
 * ============================================================================
 *
 * PICTURE (a realistic high-write pipeline, top to bottom):
 *
 *   1. INGESTION LAYER:
 *      [Load Balancer] -> [Rate Limiter] -> [API Server 1] [API Server 2]
 *
 *   2. BUFFERING LAYER:
 *      -> [Kafka: partitioned by key, replication factor 3]
 *
 *   3. PROCESSING LAYER:
 *      -> [Consumer Group: Real-Time] and [Consumer Group: Batch Analytics]
 *
 *   4. STORAGE LAYER:
 *      -> [Shard 1: Cassandra] [Shard 2: Cassandra] [Shard N: Cassandra]
 *
 *   5. READ LAYER (CQRS-style projections):
 *      -> [Elasticsearch: search] [Redis: real-time] [ClickHouse: analytics]
 *
 * HOW DATA FLOWS THROUGH IT:
 *
 *   1. INGEST:  Load balancer spreads requests, rate limiter blocks
 *      abuse, API servers validate and accept writes quickly.
 *   2. BUFFER:  Kafka absorbs spikes. Partitioning lets you process
 *      things in parallel. Replication keeps it durable.
 *   3. PROCESS: Consumer groups work in parallel, batch writes to
 *      storage, and handle retries idempotently.
 *   4. STORE:   Cassandra (LSM-based) handles the raw write volume.
 *      Sharding spreads the load. Replication protects durability.
 *   5. PROJECT: Events get turned into read-optimized views —
 *      Elasticsearch for search, Redis for real-time lookups,
 *      ClickHouse for heavy analytics queries.
 *
 *
 * ============================================================================
 * BIG TAKEAWAYS (remember these if nothing else)
 * ============================================================================
 *
 *   1. Every accepted write needs a CLEAR durability promise. Decide
 *      upfront: does "success" mean memory-only, a durable queue, local
 *      disk, quorum replication, or multi-region confirmation?
 *
 *   2. Batching is usually your FIRST move. Combining many small writes
 *      into one bigger operation cuts overhead dramatically — reach for
 *      it early, especially for independent events like logs or metrics.
 *
 *   3. Async processing separates "accept" from "actually process."
 *      Durable queues absorb spikes and let you scale consumers
 *      independently — just remember to handle delayed visibility and
 *      retries properly.
 *
 *   4. LSM trees are built for writing fast. Databases like Cassandra,
 *      RocksDB, and similar engines use append-friendly structures that
 *      trade some read/compaction cost for much better write throughput.
 *
 *   5. Shard when a single node truly isn't enough anymore. Pick your
 *      shard key carefully — a bad choice creates "hot shards" that
 *      defeat the whole point of sharding.
 *
 *   6. CQRS lets you optimize reads and writes SEPARATELY, when they
 *      genuinely have different needs — at the cost of extra complexity.
 *
 *   7. Flow control (backpressure, rate limits, load shedding, graceful
 *      degradation) prevents small overloads from becoming full-blown
 *      cascading failures. Always have SOME of this in place.
 *
 *   8. Make write processing idempotent everywhere. In distributed
 *      systems, duplicates WILL happen — design so processing the same
 *      thing twice is always harmless.
 *
 * ============================================================================
 */

In [ ]:
/*
================================================================================
   HIGH WRITE TRAFFIC SYSTEM DESIGN — MOCK INTERVIEW PREP (HINGLISH)
   Topic: Handling High Write Traffic (IoT, Logging, Analytics, Trading, etc.)
================================================================================

   Yeh document ek REAL mock interview ki tarah likha gaya hai. Isme
   INTERVIEWER sawaal poochta hai, CANDIDATE model answer deta hai, aur
   beech-beech mein follow-up + trap questions bhi aate hain. Har concept ke
   saath ek simple ASCII diagram diya gaya hai taaki visually samajh aaye.

   Goal: Is document ko 2-3 baar padhne ke baad, aap confidently "Design a
   system that handles high write traffic" wala interview de sakein.

================================================================================
ROUND 0: INTRO — TOPIC KYU IMPORTANT HAI (Quick Context)
================================================================================

INTERVIEWER: Chaliye shuru karte hain. Aaj hum baat karenge "High Write
Traffic" handle karne ke baare mein. Pehle mujhe batao — yeh problem alag
kyu hai normal system design se?

CANDIDATE: Sir, dekhiye, jyadatar systems "read-heavy" hote hain — jaise
Instagram, jaha ek post 1 baar likha jata hai but lakhon baar padha jata
hai. Wahan hum cache laga sakte hain, replicas bana sakte hain, load
easily distribute ho jata hai.

Lekin write-heavy systems mein — jaise GPS location updates, IoT sensor
data, stock trades, server logs — har request ek NAYI cheez create kar
rahi hai. Read jaisa "same data baar baar do" wala shortcut yahan nahi
chalta. Har write ko:
  1. Kahin store karna hai (durability)
  2. Kisi order mein rakhna hai (agar zaroorat ho)
  3. Kisi node ke fail hone par bhi safe rehna hai

Isliye writes scale karna genuinely harder hai reads scale karne se.

INTERVIEWER: Accha, ek simple example do jisse yeh farak clear ho jaye.

CANDIDATE: Bilkul. Maan lijiye Uber jaisa app hai. Read side pe — "nearby
drivers dikhao" — hum cache se serve kar sakte hain, thoda stale data bhi
chalega. Lekin write side pe — "driver ki current location update karo" —
yeh har 4-5 second mein har driver se aa raha hai, aur agar hum yeh write
miss kar dein, toh app galat location dikhayega. Isko ignore nahi kar
sakte.


================================================================================
ROUND 1: PROBLEM UNDERSTANDING
================================================================================

INTERVIEWER: Chalo, maan lo aapko design karna hai ek system jo IoT
sensors se aane wali data ko handle kare — jaise smart meters ya fleet
tracking (Tesla jaisa). Kaise shuru karoge?

CANDIDATE: Sir, main pehle problem ko samajhna chahunga — kitna scale hai,
kya guarantees chahiye, waghera. Main kuch clarifying questions poochna
chahunga uske baad hi design pe jaunga.

INTERVIEWER: Theek hai, poocho.

  (Yeh clarifying questions Round 2 mein detail mein cover honge)

CANDIDATE: Bas ek high-level samajh ke liye — main soch raha hoon ki yeh
system "ingest karo, store karo, aur baad mein analyze/query karne layak
banao" wala pipeline hai. Iska matlab teen core cheezein honi chahiye:

  1. INGESTION — bahut saari writes ko fast accept karna
  2. DURABILITY — accept karne ke baad data khona nahi chahiye
  3. QUERYABILITY — baad mein us data ko efficiently padh sakein

┌──────────────────────────────────────────────────────────────┐
│                    HIGH-LEVEL MENTAL MODEL                    │
│                                                                │
│   Devices/Clients          System                Consumers    │
│   ┌───────────┐      ┌──────────────────┐      ┌───────────┐  │
│   │  Sensor 1  │      │                  │      │ Dashboard │  │
│   │  Sensor 2  │─────▶│  INGEST → STORE  │─────▶│  Alerts   │  │
│   │  Sensor N  │      │                  │      │  Reports  │  │
│   └───────────┘      └──────────────────┘      └───────────┘  │
│                                                                │
│   Millions of writes/sec        Must not lose data            │
└──────────────────────────────────────────────────────────────┘

INTERVIEWER: Good. Ab thoda scale ka sense karo. Agar 10 million devices
hain, har 5 second mein ek reading bhej rahe hain, aur har message 500
bytes ka hai — toh numbers kya banenge?

CANDIDATE: Chaliye calculate karte hain, sir:

    10,000,000 devices / 5 seconds  =  2,000,000 writes/second

    2,000,000 writes/sec × 500 bytes  =  1,000,000,000 bytes/sec
                                        = 1 GB/sec ingestion

    1 GB/sec × 86,400 seconds/day  ≈  86 TB/day storage

INTERVIEWER: Ek normal PostgreSQL database yeh handle kar sakta hai?

CANDIDATE: Nahi sir, bilkul nahi. Ek achhe se tune kiya hua relational
database tens of thousands of writes/sec tak jaa sakta hai — matlab
~10,000-50,000 range mein, hardware aur config pe depend karta hai. Lekin
yahan hume 2 million writes/sec chahiye — yeh gap 40-100x ka hai. Iska
matlab single-node relational DB is problem ke liye directly fit nahi
hoga. Hume specialized techniques chahiye honge — batching, async
processing, sharding, write-optimized storage — jo hum aage discuss
karenge.

  TRAP QUESTION:
INTERVIEWER: Toh seedha keh do — PostgreSQL is bekar hai is use-case ke
liye, hum sirf Cassandra use karenge?

CANDIDATE: Sir, main aisa absolute statement nahi karunga. PostgreSQL
"bekar" nahi hai — yeh ek different trade-off ke liye designed hai:
strong consistency, transactions, rich indexing. Cassandra jaisa LSM-based
system high write throughput ke liye better fit hai, lekin uske apne
trade-offs hain — jaise eventual consistency aur read amplification.

Mera decision depend karega requirements pe: agar mujhe ACID transactions
chahiye har write pe, PostgreSQL (sharded) bhi ek option hai. Agar pure
high-throughput append-only ingestion chahiye, LSM-based store better
hoga. Toh main "PostgreSQL bekar hai" nahi kahunga — main kahunga "is
scale aur is access pattern ke liye LSM-based store better trade-off
deta hai."


================================================================================
ROUND 2: CLARIFYING QUESTIONS
================================================================================

INTERVIEWER: Chalo ab poochte hain — agar tumhe yeh system design karna
ho, tum mujhse kya kya poochoge shuru mein?

CANDIDATE: Sir, main yeh 5 sawaal zaroor poochunga, kyunki inka jawab
poora architecture decide karta hai:

┌─────────────────────────────────────────────────────────────────┐
│  Q1: Kya hum acknowledged write khone ka risk le sakte hain?      │
│      → Decide karta hai: memory buffer OK hai, ya durable queue   │
│        chahiye, ya sync replication chahiye                      │
├─────────────────────────────────────────────────────────────────┤
│  Q2: Kya ordering matter karta hai?                               │
│      → Per-key ordering chahiye? Single leader chahiye? Ya        │
│        partitioned log se kaam chal jayega?                       │
├─────────────────────────────────────────────────────────────────┤
│  Q3: Write ke turant baad usko read hona zaroori hai kya?          │
│      → Strong consistency vs eventual consistency (analytics)     │
├─────────────────────────────────────────────────────────────────┤
│  Q4: Data ka natural partition key kya hai?                       │
│      → device_id? user_id? timestamp? Yeh sharding decide karega  │
├─────────────────────────────────────────────────────────────────┤
│  Q5: Traffic steady hai ya spiky (bursty)?                        │
│      → Kitna buffering aur backpressure chahiye                  │
└─────────────────────────────────────────────────────────────────┘

INTERVIEWER: Chalo main jawab deta hoon — yeh ek fleet tracking system
hai. Location updates thodi der ke liye khone se koi bada nuksan nahi,
lekin ordering matter karta hai per-vehicle basis pe. Reads real-time hone
chahiye (dashboard live map dikhata hai). Partition key vehicle_id ho
sakta hai. Traffic mostly steady hai, lekin traffic jam ya event ke time
spike aa sakta hai.

CANDIDATE: Perfect, sir, yeh clarity bahut helpful hai. Toh main architecture
is tarah plan karunga:

  - Chunki "kuch loss tolerate ho sakta hai" — hum memory buffer / async
    queue use kar sakte hain, har write pe synchronous multi-region commit
    ki zaroorat nahi.
  - Chunki "per-vehicle ordering chahiye" — main queue ko vehicle_id se
    partition karunga (jaise Kafka partition by key), taaki ek vehicle ki
    saari updates order mein hi process ho.
  - Chunki "reads real-time chahiye" — main ek fast read-path banaunga,
    jaise Redis mein latest location cache karke, taaki dashboard fresh
    data dekhe.
  - Partition key: vehicle_id.
  - Traffic spiky ho sakta hai, toh main queue-based buffering aur
    backpressure zaroor rakhunga.

  TRAP QUESTION:
INTERVIEWER: Achha ek baat batao — agar loss tolerate ho sakta hai, toh
kya hum seedha in-memory queue use kar lein, Kafka jaisi heavy cheez ki
zaroorat hi nahi?

CANDIDATE: Sir, "kuch loss tolerate ho sakta hai" ka matlab yeh nahi ki
"bahut zyada loss theek hai." Fleet tracking mein agar ek server crash ho
jaye aur in-memory queue mein pada hua saara data (jo abhi tak persist
nahi hua) gayab ho jaye — toh woh ek badi outage jaisa lagega, especially
peak traffic ke time.

Main yahan durable queue (jaise Kafka, jo disk pe replicate karta hai)
prefer karunga kyunki:
  - Yeh crash ke baad bhi data retain karta hai
  - Consumer scaling asaan hoti hai
  - Replay possible hai agar downstream process fail ho jaye

Pure in-memory queue sirf tab use karunga jab "second-level" precision ka
loss bhi fully acceptable ho — jaise ek non-critical debug metrics stream.
Yahan fleet tracking business-critical hai, isliye main durable queue ka
extra cost justify kar sakta hoon.


================================================================================
ROUND 3: CORE APPROACHES (with Pros/Cons)
================================================================================

INTERVIEWER: Chalo ab actual approaches discuss karte hain. Batao, high
write traffic handle karne ke liye kaunse tools/techniques available
hain?

CANDIDATE: Sir, main inhe ek toolkit ki tarah dekhta hoon — inme se koi
ek "silver bullet" nahi hai, real systems inko combine karte hain:

┌────────────────────────────────────────────────────────────────┐
│                    WRITE SCALING TOOLKIT                        │
│                                                                  │
│   1. Batching & Buffering    → per-write overhead kam karo       │
│   2. Async Processing        → accept aur process alag karo      │
│   3. Sharding                → load ko multiple DB mein baanto   │
│   4. Write-Optimized Storage → LSM trees, append-only logs       │
│   5. Event Sourcing / CQRS   → read/write model alag karo        │
│   6. Backpressure/RateLimit  → overload se system ko bachao      │
└────────────────────────────────────────────────────────────────┘

Main inhe ek-ek karke discuss karta hoon.

--------------------------------------------------------------------
APPROACH 1: BATCHING & BUFFERING
--------------------------------------------------------------------

CANDIDATE: Sabse simple aur pehla lever hai batching. Idea yeh hai ki
ek-ek write bhejne ke bajaye, hum multiple writes ko ek buffer mein jama
karte hain aur ek saath flush karte hain.

┌───────────────────────────────────────────────────────┐
│              CLIENT-SIDE BATCHING                       │
│                                                          │
│   App ──▶ [Write Buffer: in-memory]                     │
│              │                                          │
│              │  Flush jab:                              │
│              │  - size limit hit ho (e.g. 1000 items)   │
│              │  - YA time limit hit ho (e.g. 100ms)      │
│              ▼                                          │
│           [Bulk write to DB]                            │
└───────────────────────────────────────────────────────┘

Iska fayda: ek single write mein ~5ms lagta hai (network RTT + parse +
transaction overhead), jisme se sirf ~0.5ms actual data write hota hai,
baaki fixed overhead hai. Agar hum 1000 rows batch karke bhejte hain, toh
total time ~55ms lagta hai — matlab per-row cost bahut kam ho jata hai.

Pros:
  - Implement karna simple hai
  - Dramatic throughput improvement, bina infrastructure badle
  - Database-level bulk methods (jaise PostgreSQL ka COPY command) available

Cons:
  - Latency trade-off — individual write turant commit nahi hota
  - Agar batch flush hone se pehle crash ho jaye, buffer mein pada data
    kho sakta hai (agar sirf in-memory buffer hai)
  - Payment jaise use-cases mein yeh delay acceptable nahi ho sakta

INTERVIEWER: Agar time-critical payment write ho, toh batching use kar
sakte ho?

CANDIDATE: Directly nahi, sir. Payment writes ko turant acknowledge aur
persist karna hota hai — user ko turant pata chalna chahiye ki transaction
successful hua ya nahi. Batching yahan latency add kar degi jo unacceptable
hai. Lekin haan, agar payment ke saath koi "side effect" write hai — jaise
ek audit log entry, ya analytics event — usko hum batch kar sakte hain,
kyunki uska turant persist hona critical nahi hai.

--------------------------------------------------------------------
APPROACH 2: ASYNCHRONOUS PROCESSING (Queue-based)
--------------------------------------------------------------------

CANDIDATE: Yeh technique "kab" write hota hai usko change karti hai.
Instead of directly database mein likhne ke, hum ek message queue (jaise
Kafka) mein publish karte hain, aur ek separate consumer usse consume
karke database mein likhta hai.

┌────────────────────────────────────────────────────────────────┐
│                  ASYNC WRITE PIPELINE                            │
│                                                                   │
│  Producers          Queue (durable)         Consumers    Storage │
│  ┌─────────┐        ┌──────────────┐       ┌─────────┐  ┌─────┐ │
│  │ API Srv1│──publish─▶│  Kafka       │──consume─▶│Worker1│─▶│ DB │ │
│  │ API Srv2│──publish─▶│  Partitioned │──consume─▶│Worker2│─▶│    │ │
│  │ API SrvN│──publish─▶│  Replicated  │──consume─▶│WorkerN│─▶│    │ │
│  └─────────┘        └──────────────┘       └─────────┘  └─────┘ │
└────────────────────────────────────────────────────────────────┘

Pros:
  - Producer turant respond kar sakta hai, DB ke slow hone ka wait nahi
  - Traffic spike queue mein absorb ho jati hai
  - Consumers independently scale ho sakte hain
  - DB down ho jaye toh bhi writes queue mein safe rehte hain (agar
    durable queue hai)

Cons:
  - Eventual consistency — write turant queryable nahi hota
  - Extra infra complexity (Kafka/SQS/RabbitMQ manage karna)
  - Duplicate delivery ho sakti hai (at-least-once), isliye idempotency
    zaroori hai
  - Consumer lag monitor karna padta hai

  TRAP QUESTION:
INTERVIEWER: Toh agar main tumse kahoon "sirf ek approach batao — sabse
best" — batching ya async queue?

CANDIDATE: Sir, main ek single "best" nahi bataunga, kyunki yeh dono
different problems solve karte hain aur real systems mein saath chalte
hain — mutually exclusive nahi hain.

  - Batching optimize karta hai "kaise" write hota hai — cost per write
    kam karta hai.
  - Async processing optimize karta hai "kab" write hota hai — accept
    aur persist ko decouple karta hai.

Ek typical high-write pipeline mein: producer pehle Kafka mein publish
karta hai (async), fir consumer Kafka se messages ko BATCH karke database
mein likhta hai. Toh dono ek saath use hote hain — ek dusre ke alternative
nahi, complementary hain.

Agar aap mujhe zabardasti ek choose karne ko bolein "sirf ek use kar
sakte ho," toh main poochunga: "kya hum thodi latency afford kar sakte
hain?" Agar haan, async queue pehle add karunga kyunki woh resilience aur
decoupling deta hai jo batching akela nahi deta.

--------------------------------------------------------------------
APPROACH 3: SHARDING
--------------------------------------------------------------------

CANDIDATE: Jab ek single database node ki capacity khatam ho jaye, hum
writes ko multiple databases (shards) mein distribute karte hain.

┌─────────────────────────────────────────────────────────┐
│                     WRITE SHARDING                        │
│                                                             │
│   App ──▶ [Shard Router] ──hash(user_id) % 4──▶            │
│                │        │        │        │                │
│                ▼        ▼        ▼        ▼                │
│           [Shard0] [Shard1] [Shard2] [Shard3]              │
│           25% load  25% load 25% load 25% load             │
└─────────────────────────────────────────────────────────┘

Pros:
  - Near-linear horizontal scaling (4 shards ≈ 4x capacity, agar no
    shared bottleneck aur even distribution ho)
  - Failure blast radius chhota hota hai (ek shard down, baaki chalte
    rehte hain)

Cons:
  - Shard key sahi choose karna critical hai — galat key se "hot shard"
    ban jata hai
  - Cross-shard transactions complex ho jaate hain (2PC ya Saga chahiye)
  - Resharding (adding/removing shards) painful ho sakta hai bina
    consistent hashing ke

INTERVIEWER: Shard key kaise choose karoge fleet tracking system mein?

CANDIDATE: Main vehicle_id use karunga, sir, kyunki:
  - Ek vehicle ki saari updates naturally ek hi shard pe jayengi
  - Load fairly evenly distribute hoga (assuming vehicles similar rate pe
    data bhejte hain)

Main timestamp ko shard key NAHI banaunga, kyunki "current time" wala
saara traffic ek hi shard pe collapse ho jayega — baaki shards idle
rahenge aur ek shard overload.

  TRAP QUESTION:
INTERVIEWER: Achha, agar ek vehicle "viral" ho jaye — jaise ek company ka
1000 trucks wala fleet ek hi vehicle_id prefix use kar raha ho — toh kya
hoga?

CANDIDATE: Achha catch hai sir. Agar single key (ya related keys ka
cluster) bahut zyada traffic generate kare, toh woh ek "hot shard" ban
jayega, chahe hashing kitni bhi achhi ho. Iske liye main:
  1. Composite shard key use karunga — jaise (vehicle_id, time_bucket) —
     taaki ek entity ka load bhi thoda spread ho sake
  2. Application level pe monitoring rakhunga taaki hot keys detect ho
     saken
  3. Zaroorat pade toh us specific high-volume entity ko manually split
     kar sakta hoon multiple sub-shards mein (jaise consistent hashing
     mein virtual nodes ka concept)

Main yeh nahi kahunga ki "sharding hamesha perfectly even hoga" — real
world mein hot keys aate hain, aur unhe handle karne ka plan hona chahiye.

--------------------------------------------------------------------
APPROACH 4: WRITE-OPTIMIZED STORAGE (LSM Trees)
--------------------------------------------------------------------

CANDIDATE: Yeh storage engine level ka decision hai. Traditional
databases (PostgreSQL, MySQL) B-Tree use karte hain, jo reads ke liye
optimized hai lekin writes mein random I/O involve karta hai.

┌──────────────────────────┐    ┌──────────────────────────┐
│   B-TREE WRITE PATH        │    │   LSM TREE WRITE PATH      │
│   (Random I/O)              │    │   (Sequential/Append)      │
│                              │    │                            │
│  Write → Find page          │    │  Write → Memtable (RAM)    │
│        → Read page (disk)   │    │        → flush when full   │
│        → Modify page        │    │        → SSTable (disk)    │
│        → Write page (disk)  │    │        → background merge  │
│        → Update indexes     │    │          & compaction       │
└──────────────────────────┘    └──────────────────────────┘

LSM Tree (Cassandra, RocksDB, ScyllaDB) writes seedha ek in-memory
memtable mein jaate hain, jo bhar jaane par disk pe ek sorted file
(SSTable) ke roop mein flush hota hai. Background process inhe merge
(compact) karta rehta hai. Yeh write path ko mostly append-oriented
banata hai — isliye ingestion ke liye bahut fast hai.

Pros:
  - Bahut high write throughput (append-friendly)
  - Sequential disk I/O, jo random I/O se fast hota hai

Cons:
  - Read amplification — ek read multiple SSTable levels check kar sakta
    hai
  - Compaction background CPU/IO consume karta hai
  - Space amplification compaction ke dauraan

INTERVIEWER: Toh kya iska matlab hum hamesha Cassandra use karein high
writes ke liye?

CANDIDATE: Nahi sir, yeh depend karta hai read pattern pe bhi. Agar
system ko strong point-lookups aur complex joins chahiye saath mein high
writes ke, toh sirf LSM-based store kaafi nahi hoga — hume CQRS jaisa
pattern chahiye hoga jaha write path LSM-optimized ho aur alag se read
projections banayi jayein. LSM tree ek "silver bullet" nahi, ek trade-off
hai — write-heavy, append-mostly workloads ke liye best fit.

--------------------------------------------------------------------
APPROACH 5: EVENT SOURCING / CQRS
--------------------------------------------------------------------

CANDIDATE: Yeh ek architectural shift hai — current state store karne ke
bajaye, hum events (state changes) store karte hain.

┌──────────────────────────────────────────────────────────┐
│                    EVENT SOURCING                          │
│                                                              │
│   UserCreated → DepositMade(+100) → WithdrawalMade(-20)      │
│                                                              │
│   Replay all events = Current Balance (80)                  │
│   Full history preserved — audit trail built-in             │
└──────────────────────────────────────────────────────────┘

CQRS (Command Query Responsibility Segregation) isse pair hota hai —
writes (commands) event store mein jaate hain, aur alag "read models"
(projections) query patterns ke hisaab se banaye jaate hain (jaise
Elasticsearch for search, Redis for real-time, ClickHouse for analytics).

Pros:
  - Write path pure append-only, bahut fast
  - Complete audit trail / history "free" mein milta hai
  - Read aur write independently scale ho sakte hain

Cons:
  - Significant complexity add hota hai
  - Projection lag — read model thoda purana ho sakta hai
  - Simple CRUD apps ke liye overkill

INTERVIEWER: Fleet tracking system mein event sourcing use karoge?

CANDIDATE: Partially, sir. Main pura Event Sourcing/CQRS system nahi
banaunga kyunki fleet tracking mein complex domain logic ya audit
requirement utni nahi jitni ki banking mein hoti hai. Lekin CQRS ka idea
— write path alag, read path alag — main zaroor use karunga: raw location
events Kafka/Cassandra mein append honge (write side), aur latest
location Redis mein project hoga fast dashboard reads ke liye (read
side). Yeh "CQRS-lite" jaisa hai, bina full event sourcing complexity ke.


================================================================================
ROUND 4: COMPARISON TABLE
================================================================================

INTERVIEWER: Chalo sab approaches ka ek quick comparison do.

CANDIDATE: Zaroor, sir:

┌─────────────────┬───────────────┬────────────────┬───────────────────┐
│ Technique         │ Solves         │ Trade-off       │ Best For           │
├─────────────────┼───────────────┼────────────────┼───────────────────┤
│ Batching          │ Per-write       │ Adds latency     │ Independent events │
│                    │ overhead        │                  │ (logs, metrics)    │
├─────────────────┼───────────────┼────────────────┼───────────────────┤
│ Async Queue       │ Spike absorption│ Eventual         │ Decoupled          │
│                    │ + resilience    │ consistency      │ producer/consumer  │
├─────────────────┼───────────────┼────────────────┼───────────────────┤
│ Sharding          │ Single-node     │ Cross-shard      │ Data that has a    │
│                    │ capacity limit  │ ops complexity   │ natural key        │
├─────────────────┼───────────────┼────────────────┼───────────────────┤
│ LSM-based Storage │ Slow write path │ Read/compaction  │ Append-heavy,      │
│                    │ (B-tree random) │ amplification    │ time-series data   │
├─────────────────┼───────────────┼────────────────┼───────────────────┤
│ Event Sourcing/   │ Read/write      │ Architectural    │ Audit-heavy,       │
│ CQRS               │ pattern mismatch│ complexity       │ multi-read-model   │
├─────────────────┼───────────────┼────────────────┼───────────────────┤
│ Backpressure/     │ Cascading       │ Rejects/delays   │ Always (defensive) │
│ Rate Limiting      │ failure         │ some requests    │                    │
└─────────────────┴───────────────┴────────────────┴───────────────────┘

  TRAP QUESTION:
INTERVIEWER: Agar mujhe sirf ek technique choose karni ho poore
architecture ke liye, toh kaunsi choose karoge?

CANDIDATE: Sir, honestly, real production system mein "sirf ek technique"
kaam nahi karti — high-write systems layered hote hain. Agar aap mujhe
"pehla step kya hoga" poochein, toh main kahunga: pehle backpressure aur
rate limiting daalo, kyunki yeh defensive hai aur baaki sab optimizations
ke bina bhi system ko crash hone se bachata hai. Uske baad batching aur
async queue add karunga, aur jab single node capacity khatam ho, tab
sharding aur write-optimized storage layer add hoga.

Toh mera answer hoga: "priority order" bata sakta hoon, "ek hi cheez"
nahi bata sakta — kyunki har technique alag failure mode solve karti hai.


================================================================================
ROUND 5: ADVANCED / HYBRID APPROACH — Full Architecture
================================================================================

INTERVIEWER: Chalo ab poora design batao is fleet tracking / IoT system
ka, end to end.

CANDIDATE: Bilkul, sir. Main isse 5 layers mein todunga:

┌──────────────────────────────────────────────────────────────────┐
│                    FULL HIGH-WRITE ARCHITECTURE                    │
│                                                                      │
│  [1] INGESTION                                                      │
│      Load Balancer → Rate Limiter → API Servers (validate & accept) │
│                              │                                       │
│                              ▼                                       │
│  [2] BUFFERING                                                       │
│      Kafka (partitioned by vehicle_id, replication factor 3)         │
│                              │                                       │
│              ┌───────────────┴───────────────┐                       │
│              ▼                                 ▼                     │
│  [3] PROCESSING                                                       │
│   Consumer Group (Real-time)          Consumer Group (Batch)          │
│   - batches small chunks                - batches large chunks        │
│   - idempotent writes                   - analytics rollups           │
│              │                                 │                       │
│              ▼                                 ▼                       │
│  [4] STORAGE                                                           │
│   Cassandra (sharded, LSM-based) ── raw event history                 │
│                                                                         │
│  [5] READ / PROJECTION LAYER (CQRS-lite)                               │
│   Redis (latest location, real-time)                                   │
│   ClickHouse (analytics, historical trends)                            │
│   Elasticsearch (search by region/fleet, optional)                     │
└──────────────────────────────────────────────────────────────────┘

Data flow:
  1. INGEST: Vehicle bhejta hai location update → API server validate
     karta hai (schema check, auth) aur turant Kafka mein publish karke
     ack return kar deta hai. Latency yahan bahut low honi chahiye.
  2. BUFFER: Kafka topic vehicle_id se partitioned hai — isse per-vehicle
     ordering guarantee milti hai, aur replication factor 3 durability
     deta hai.
  3. PROCESS: Ek consumer group real-time projection banata hai (Redis
     mein "last known location" update karta hai) — yeh chhote batches
     mein fast process hota hai. Doosra consumer group batch analytics ke
     liye bade batches mein Cassandra/ClickHouse mein likhta hai.
  4. STORE: Cassandra full raw history store karta hai, sharded by
     vehicle_id, jisse write throughput high rehta hai (LSM-based).
  5. PROJECT: Read-side needs ke hisaab se — dashboard Redis se live data
     leta hai (fast), reports ClickHouse se leta hai (analytical
     queries).

Idempotency: Har location event ke saath ek unique event_id (jaise
vehicle_id + sequence_number) hoga. Consumer processing se pehle check
karega ki yeh event already process ho chuka hai ya nahi (dedup table ya
Redis set mein), taaki Kafka ke at-least-once delivery se duplicate
processing na ho.

┌─────────────────────────────────────────────┐
│         IDEMPOTENT CONSUMER PATTERN            │
│                                                 │
│   Message arrives                              │
│        │                                       │
│        ▼                                       │
│   Already processed? ──Yes──▶ Skip, ack anyway │
│        │No                                     │
│        ▼                                       │
│   Process + Store event_id                     │
│        │                                       │
│        ▼                                       │
│   Acknowledge to Kafka                          │
└─────────────────────────────────────────────┘

  TRAP QUESTION:
INTERVIEWER: Agar main tumhe bolun ki Kafka use nahi kar sakte (budget/
infra constraint), sirf Redis available hai — kya karoge?

CANDIDATE: Sir, is case mein main Redis Streams ya simple Redis List ko
queue jaisa use karunga, lekin main ek trade-off explicitly bata dunga:
Redis primarily in-memory hai, toh durability guarantee Kafka jaisi
strong nahi hogi — agar Redis persistence (AOF/RDB) properly configure na
ho, crash ke case mein data loss ho sakta hai.

Main mitigate karunga:
  - Redis persistence (AOF with appendfsync) enable karke
  - Redis replication (at least 1 replica) rakh ke
  - Downstream consumer ko jaldi se jaldi durable storage mein persist
    karne ka design banake, taaki Redis mein data zyada der na ruke

Lekin main clearly candidly bataunga interviewer ko: "yeh Kafka jitna
durable nahi hoga, aur agar loss zero-tolerance hai, toh main isse
recommend nahi karunga — main durable queue infra ke liye push karunga."


================================================================================
ROUND 6: EDGE CASES
================================================================================

INTERVIEWER: Chalo kuch edge cases discuss karte hain.

EDGE CASE 1 — Consumer Lag / Falling Behind:

INTERVIEWER: Agar consumers producers se slow ho jayein aur lag badhta
jaye, toh kya hoga aur kaise detect karoge?

CANDIDATE: Sir, "consumer lag" hi woh metric hai jo isse expose karta
hai — matlab latest offset likha gaya (jo producer ne publish kiya) aur
offset jo consumer ne process kar liya hai, unka difference.

┌───────────────────────────────────────────┐
│  latest offset written:      1,000,000       │
│  consumer committed offset:    940,000       │
│  ─────────────────────────────────────       │
│  LAG = 60,000 messages (consumer peeche hai)  │
└───────────────────────────────────────────┘

Main is par alert lagaunga. Agar lag steadily badh raha hai, toh main:
  1. Consumers ki count badhaunga (up to partition count — ek partition
     ek hi consumer process karta hai ek group mein)
  2. Batch size increase karunga taaki per-batch overhead kam ho
  3. Upstream backpressure apply karunga agar consumers genuinely max
     capacity pe hain

EDGE CASE 2 — Hot Partition / Hot Shard:

INTERVIEWER: Agar ek device bahut zyada data bhejne lage (malfunction ya
bug ki wajah se), aur uska partition/shard overload ho jaye?

CANDIDATE: Sir, main rate limiting per device_id lagaunga, taaki ek
misbehaving device poore partition ko overload na kare. Main monitoring
bhi rakhunga jo per-key throughput track kare, taaki anomaly detect ho
sake. Extreme case mein, main us device ko temporarily throttle ya block
kar sakta hoon (circuit breaker jaisa).

EDGE CASE 3 — Duplicate Writes:

INTERVIEWER: Agar network retry ki wajah se ek hi write do baar aa jaye,
kya hoga?

CANDIDATE: Isko main idempotency se handle karunga — jaisa maine pehle
bataya, ek unique event_id (ya vehicle_id + timestamp/sequence) generate
hoga client side pe, aur server/consumer dedup check karega processing se
pehle. Agar duplicate hai, toh silently skip karke success return kar
denge (kyunki result same hi hoga).

EDGE CASE 4 — Leader Failure Mid-Write:

INTERVIEWER: Agar leader-based system mein (jaise Kafka partition leader
ya PostgreSQL primary) leader crash ho jaye jab write in-flight hai?

CANDIDATE: Sir, is case mein kuch cheezein ho sakti hain:
  - Agar write leader tak pahunch gaya tha aur usne acknowledge kar diya
    tha (aur replicate bhi ho chuka tha), toh naye leader ke paas woh
    data hoga — safe hai.
  - Agar write leader tak pahuncha but replicate hone se pehle hi crash
    ho gaya, toh woh write potentially lost ho sakta hai (agar
    acknowledgment replica confirm hone se pehle diya gaya tha — yeh ek
    design mistake hai).

Isliye main "acknowledge before durable" wala anti-pattern avoid karunga.
Main ensure karunga ki ack tabhi bheja jaye jab write kam se kam quorum
replicas tak pahunch chuka ho (W acks jahan zaroorat ho). Client ko bhi
main retry logic dunga (with idempotency) taaki leader failover ke
dauraan bhi unacknowledged writes safely retry ho sakein.

EDGE CASE 5 — Cross-Shard Atomic Operation:

INTERVIEWER: Agar do alag shards ke beech ek atomic operation chahiye ho
(jaise money transfer)?

CANDIDATE: Sir, best approach hai design karna aise ki common operations
single-shard ke andar hi rahein (jaise ek account aur uska ledger same
shard pe rakhna). Jab cross-shard avoid nahi ho sakta:
  - Saga pattern use karunga — eventual consistency ke saath, compensating
    transactions ke through
  - 2PC (Two-Phase Commit) — strong consistency deta hai lekin performance
    slow hoti hai locks ki wajah se, high write-throughput system ke liye
    generally avoid karna better hai

  TRAP QUESTION:
INTERVIEWER: Toh seedha bolo — Saga hamesha better hai 2PC se?

CANDIDATE: Nahi sir, aisa universal statement sahi nahi hoga. Saga
availability aur performance ke liye better hai, lekin eventual
consistency deta hai aur implementation complex hoti hai (compensating
transactions likhne padte hain, jo galat hone par data inconsistency ka
risk create kar sakte hain).

2PC strong consistency deta hai — agar business requirement bilkul
strict atomicity maange (jaise banking ka core ledger transfer), aur
scale utna extreme na ho, 2PC ek valid choice ho sakta hai.

Mera decision depend karega: high-throughput, eventually-consistent
domain → Saga. Low-throughput, strict-consistency-critical domain → 2PC
ya phir design hi aisa karo ki cross-shard transaction ki zaroorat na
pade.


================================================================================
ROUND 7: SCALING / PERFORMANCE DEEP DIVE
================================================================================

INTERVIEWER: Chalo scaling pe deep dive karte hain. Agar traffic 10x badh
jaye achanak (jaise ek naya bada client onboard ho gaya), kya karoge?

CANDIDATE: Sir, main step by step dekhunga system ke har layer ko:

┌────────────────────────────────────────────────────────────┐
│  Layer            │  Scaling Lever                            │
├────────────────────────────────────────────────────────────┤
│  Ingestion (API)   │  Add more API server instances            │
│                    │  (stateless, horizontally scalable)       │
├────────────────────────────────────────────────────────────┤
│  Queue (Kafka)     │  Add more partitions (needs re-planning   │
│                    │  since partition count usually only       │
│                    │  increases, not decreases easily)          │
├────────────────────────────────────────────────────────────┤
│  Consumers         │  Add more consumer instances (up to        │
│                    │  partition count)                          │
├────────────────────────────────────────────────────────────┤
│  Storage (shards)  │  Add more shards + rebalance via           │
│                    │  consistent hashing (with virtual nodes)   │
├────────────────────────────────────────────────────────────┤
│  Read layer        │  Add read replicas / cache nodes           │
└────────────────────────────────────────────────────────────┘

Consistent Hashing ka role:

┌─────────────────────────────────────────────┐
│  Naive Modulo (key % N):                       │
│    3 shards → 4 shards  =  ~75% keys move       │
│                                                 │
│  Consistent Hashing (with virtual nodes):       │
│    3 shards → 4 shards  =  ~25% keys move       │
│    (only new shard's slice, spread across many  │
│     neighbors instead of dumping on one)        │
└─────────────────────────────────────────────┘

Main consistent hashing (virtual nodes ke saath) use karunga taaki
resharding ka operational pain kam ho.

INTERVIEWER: Durability aur latency ka trade-off kaise decide karoge?

CANDIDATE: Sir, main isko ek "durability spectrum" ki tarah dekhta hoon:

┌───────────────────────────────────────────────────────────┐
│  Level         │  Method               │  Latency │ Survives  │
├───────────────────────────────────────────────────────────┤
│  1 (lowest)    │  Memory only          │  ~1μs    │  Nothing   │
│  2             │  Async disk write     │  ~100μs  │  Process   │
│                │                        │          │  crash     │
│  3             │  Sync disk (fsync)    │  ~1-10ms │  Machine   │
│                │                        │          │  crash     │
│  4             │  Sync + replica ack   │  ~10-50ms│  Machine   │
│                │                        │          │  loss      │
│  5 (highest)   │  Multi-region sync    │  ~50-200 │  Datacenter│
│                │                        │  ms      │  loss      │
└───────────────────────────────────────────────────────────┘

Fleet tracking jaisa system, jaha thoda loss tolerate ho sakta hai, main
Level 2 ya Level 3 pe operate karunga — kyunki latency-sensitive hai
(real-time dashboard). Agar yeh trading system hota, main Level 4 ya 5
choose karta, kyunki ek transaction bhi khona unacceptable hai.

  TRAP QUESTION:
INTERVIEWER: Toh sabse zyada durability wala option (Level 5) hamesha
"safest choice" hai na, isko hi default kyun nahi bana dete?

CANDIDATE: Sir, "safest" ka matlab hamesha "best" nahi hota engineering
mein. Level 5 (multi-region sync) 50-200ms latency add karta hai har
write pe. Agar system 2 million writes/sec handle kar raha hai aur har
write itni latency leta hai, toh throughput drastically gir jayega ya
bahut zyada parallel connections chahiye honge, jo cost aur complexity
dono badha dega.

Har use case ka apna "acceptable loss" hota hai. Zabardasti sabse high
durability lagana ek "over-engineering" trap hai — jo latency aur cost
dono ko unnecessarily badha deta hai jab actual requirement usse kam
strict thi. Sahi approach hai: requirement se match karna, na ki hamesha
sabse extreme option choose karna.

INTERVIEWER: Rate limiting aur backpressure mein kya farak hai? Log
confuse ho jaate hain isme.

CANDIDATE: Achha sawaal hai sir, yeh dono alag purpose serve karte hain:

┌──────────────────────────────────────────────────────────┐
│  Rate Limiting                  │  Backpressure               │
├──────────────────────────────────────────────────────────┤
│  "Yeh caller apna quota          │  "System abhi overloaded    │
│   cross kar chuka hai"           │   hai, sabko slow karo"     │
│                                   │                             │
│  Fairness/abuse control          │  System self-protection      │
│                                   │                             │
│  Per-client/per-tenant limit      │  Global flow control         │
│                                   │  (queue full → signal        │
│                                   │  upstream to slow down)      │
└──────────────────────────────────────────────────────────┘

Rate limiting kehta hai "tum bahut zyada bhej rahe ho, tumhari quota khatam
hui." Backpressure kehta hai "poora system capacity pe hai, chahe koi bhi
bheje, sabko slow karna padega." Dono zaroori hain — rate limiting fairness
ke liye, backpressure survival ke liye.


================================================================================
ROUND 8: CLOSING SUMMARY
================================================================================

INTERVIEWER: Chalo, is design ko ek 2-minute summary mein wrap up karo,
jaise aap final round mein interviewer ko sunate hain.

CANDIDATE: Bilkul sir. Toh overall, high write traffic handle karne ke
liye maine yeh approach liya:

  1. Pehle requirements clarify kiye — loss tolerance, ordering,
     consistency needs, partition key, aur traffic pattern.
  2. Ingestion layer banaya jo fast, lightweight validation ke saath
     writes accept karta hai aur rate limiting + backpressure se protect
     hota hai.
  3. Ek durable, partitioned queue (Kafka) use kiya buffering ke liye,
     jo spikes absorb karta hai aur per-key ordering deta hai.
  4. Consumers ko batching ke saath likha, jo idempotent processing
     karte hain (duplicate-safe).
  5. Storage ke liye LSM-based, sharded database (Cassandra) choose kiya
     jo high write throughput ke liye optimized hai.
  6. Read side ko CQRS-lite pattern se alag kiya — real-time ke liye
     Redis, analytics ke liye ClickHouse.
  7. Poore system mein consistent hashing use kiya resharding pain kam
     karne ke liye, aur consumer lag jaisi metrics monitor ki operational
     health ke liye.

Sabse important baat jo main highlight karna chahunga: har decision ek
trade-off tha, "ek best solution" nahi. Main hamesha requirement ke
against decision justify karta hoon, na ki ek fixed template follow
karta hoon.


================================================================================
RAPID FIRE REVISION SECTION
================================================================================

Q1: Reads scale karna writes scale karne se aasan kyun hai?
A1: Reads mein cache, replicas, stale data allow kar sakte hain. Writes
    mein har write ko ordered/durable/valid banana padta hai — koi
    shortcut nahi.

Q2: Batching kyu kaam karti hai?
A2: Fixed per-write overhead (network, parsing, commit) ko many rows mein
    amortize kar deti hai.

Q3: Write-behind pattern ka risk kya hai?
A3: Agar fast buffer (jaise Redis) persist hone se pehle crash ho jaye,
    acknowledged writes kho sakte hain.

Q4: Kafka partition kyu important hai?
A4: Per-partition ordering deta hai, aur parallel consumption allow karta
    hai (scale horizontally).

Q5: Idempotency kyu zaroori hai queue-based systems mein?
A5: Queues usually "at-least-once" delivery dete hain — duplicate
    messages aa sakte hain. Idempotent processing ensure karta hai ki
    duplicate se galat result na aaye.

Q6: Consumer lag kya hai?
A6: Latest offset aur consumer ke committed offset ka difference — batata
    hai ki consumer producer se kitna peeche hai.

Q7: B-Tree vs LSM Tree — main difference?
A7: B-Tree: in-place update, random I/O, strong reads. LSM Tree: append-
    oriented (memtable → SSTable), fast writes, but read/compaction
    overhead.

Q8: Sharding mein sabse common mistake kya hai?
A8: Galat shard key choose karna (jaise timestamp) jo "hot shard" bana
    deta hai.

Q9: Consistent hashing kyu use karte hain?
A9: Resharding ke time minimum data movement ke liye (naive modulo mein
    ~75% keys move hote hain, consistent hashing mein ~25%).

Q10: Virtual nodes kis problem ko solve karte hain?
A10: Single point ring assignment se load unevenly distribute hota hai;
     virtual nodes ek physical shard ko multiple ring points dete hain,
     load evenly spread karne ke liye.

Q11: Event Sourcing kya hai?
A11: Current state store karne ke bajaye, saari state-changing events
     store karo. Current state events replay karke derive hoti hai.

Q12: CQRS kya solve karta hai?
A12: Read aur write models ko separate karta hai, taaki dono
     independently optimize/scale ho sakein.

Q13: Rate Limiting vs Backpressure?
A13: Rate limiting = per-caller quota enforcement (fairness). Backpressure
     = system-wide flow control jab overloaded ho (survival).

Q14: Load shedding kis order mein requests drop karni chahiye?
A14: Depends: priority-based (protect critical traffic), age-based
     (already-stale requests), ya client-based (over-quota clients pehle).

Q15: R + W > N ka matlab kya hai quorum systems mein?
A15: Yeh guarantee karta hai ki read set aur write set overlap karein,
     taaki read hamesha latest write dekhe (strong consistency achieve
     hoti hai quorum ke through).

Q16: Durability ka sabse extreme (Level 5) option hamesha use karna
     chahiye kya?
A16: Nahi — yeh highest latency bhi deta hai. Requirement ke hisaab se
     durability level choose karo, hamesha maximum nahi.

Q17: Sabse pehla lever kya try karna chahiye high writes ke liye?
A17: Batching — simplest, sabse kam infra change, immediate improvement.

Q18: LSM tree databases ke examples?
A18: Cassandra, ScyllaDB, RocksDB, ClickHouse, InfluxDB.

Q19: Cross-shard atomic transaction ka best solution?
A19: Avoid karo agar possible ho (data model aisa design karo ki common
     ops single-shard mein rahein). Agar avoid na ho, Saga (eventual) ya
     2PC (strong, slower) use karo.

Q20: Graceful degradation ka example do.
A20: Heavy load mein core write (jaise DB write) preserve karo, non-
     critical side-effects (jaise notifications, analytics) temporarily
     skip/defer kar do.

================================================================================
END OF DOCUMENT — Best of luck for your interview!
================================================================================
*/